# Phi-2 QLoRA fine-tuning for HEA Q&A (corrected)

Corrections relative to the first version of this notebook:
1. **Answer supervision.** Each example is `prompt + answer + EOS`; prompt tokens are masked with `-100`, so the loss is computed on the answer only. The earlier version stored the answer in `labels` but then used `DataCollatorForLanguageModeling`, which overwrote `labels` with a copy of the prompt — the model was never trained on the answers.
2. **Leakage-free split.** Train/validation/test are split by *context* (alloy record), produced by `evaluation/make_splits.py` (seed 42: 73/15/15 contexts, 2,248/520/624 questions). A random split by question would put a paraphrase of every test question into training.
3. **Evaluation.** Exact match, token F1, BLEU and ROUGE-L on the held-out test contexts, with gold context and with contexts returned by the hybrid retriever (end-to-end RAG), with 95% context-level bootstrap intervals.

Runs on a Colab T4 (≈1 h). The CPU equivalent used for the reported numbers is `evaluation/gen_eval.py --model phi2`.

In [ ]:
!pip -q install transformers==4.46.3 peft==0.13.2 bitsandbytes accelerate==1.2.1 sentence-transformers==3.2.1 faiss-cpu scikit-learn rouge-score sacrebleu
!git clone -q https://github.com/KriteshKG/HEA_LLM.git
%cd HEA_LLM/evaluation
!python make_splits.py && python retrieval_eval.py

In [ ]:
import json, math, random, sys, time
import numpy as np, torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, get_linear_schedule_with_warmup
import bitsandbytes as bnb
from gen_eval import examples, phi_prompt, score, with_retrieved
from common import RESULTS, SEED, load_split, load_squad

random.seed(SEED); torch.manual_seed(SEED)
train, val, test = (examples(load_split(s)) for s in ("train", "val", "test"))
print(len(train), len(val), len(test))

In [ ]:
MODEL = "microsoft/phi-2"
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
tok.padding_side = "left"
bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb_cfg, device_map={"": 0})
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.1, bias="none", task_type="CAUSAL_LM",
                                         target_modules=["q_proj", "k_proj", "v_proj", "dense"]))
model.print_trainable_parameters()

In [ ]:
MAX_LEN = 1024

def encode(x):
    p = tok(phi_prompt(x["context"], x["question"])).input_ids
    a = tok(" " + x["answer"]).input_ids + [tok.eos_token_id]
    p = p[-(MAX_LEN - len(a)):]
    return p + a, [-100] * len(p) + a          # loss on answer + EOS only

def collate(batch):
    enc = [encode(x) for x in batch]
    n = max(len(i) for i, _ in enc)
    ids = torch.tensor([[tok.eos_token_id] * (n - len(i)) + i for i, _ in enc])
    lab = torch.tensor([[-100] * (n - len(l)) + l for _, l in enc])
    att = (torch.arange(n)[None, :] >= (n - torch.tensor([len(i) for i, _ in enc]))[:, None]).long()
    return ids.cuda(), att.cuda(), lab.cuda()

# sanity check: the supervised tokens decode to the answer
ids, att, lab = collate(train[:1])
print(repr(tok.decode(ids[0][lab[0] != -100])), "<-", repr(train[0]["answer"]))

In [ ]:
EPOCHS, BS, ACCUM, LR = 1, 4, 4, 2e-4
opt = bnb.optim.PagedAdamW8bit([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.001)
steps = EPOCHS * math.ceil(len(train) / (BS * ACCUM))
sched = get_linear_schedule_with_warmup(opt, int(0.03 * steps), steps)
model.gradient_checkpointing_enable()
model.train()
for ep in range(EPOCHS):
    random.Random(SEED + ep).shuffle(train)
    for i, s in enumerate(range(0, len(train), BS)):
        ids, att, lab = collate(train[s:s + BS])
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(input_ids=ids, attention_mask=att, labels=lab).loss / ACCUM
        loss.backward()
        if (i + 1) % ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
        if i % 50 == 0:
            print(ep, i, round(loss.item() * ACCUM, 4))
model.save_pretrained("phi2-lora-finetuned")
tok.save_pretrained("phi2-lora-finetuned")

In [ ]:
@torch.no_grad()
def predict(data, bs=16):
    model.eval(); out = []
    for s in range(0, len(data), bs):
        enc = tok([phi_prompt(x["context"], x["question"]) for x in data[s:s + bs]], return_tensors="pt",
                  padding=True, truncation=True, max_length=MAX_LEN).to("cuda")
        gen = model.generate(**enc, max_new_tokens=24, do_sample=False, pad_token_id=tok.eos_token_id)
        out += [t.strip().split("\n")[0].strip() for t in tok.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)]
    return out

print("val", score(val, predict(val)))

In [ ]:
ctx_text = {c["cid"]: c["context"] for c in load_squad()}
retrieved = json.load(open(RESULTS / "retrieved_contexts_hybrid.json"))["test"]
results = {}
for setting, data in [("gold", test), ("top1", with_retrieved(test, retrieved, ctx_text, 1)),
                      ("top3", with_retrieved(test, retrieved, ctx_text, 3))]:
    results[setting] = score(data, predict(data))
    print(setting, {k: v for k, v in results[setting].items() if not k.endswith("ci")})
json.dump(results, open(RESULTS / "generation_phi2_qlora_gpu.json", "w"), indent=2)